# Глава 13. RAG vs Fine-tuning

## 13.1. Сравнительный анализ подходов

Адаптация больших языковых моделей к специфическим задачам представляет собой многогранную проблему, требующую выбора оптимальной стратегии из множества доступных подходов. Современная практика машинного обучения предлагает несколько фундаментально различных методов, каждый из которых обладает уникальными характеристиками, преимуществами и ограничениями.

### Retrieval‑Augmented Generation

Это способ сделать языковую модель умнее. Модель не меняется, но перед ответом она ищет нужную информацию во внешней базе знаний.

**Как это работает**
- Модель получает вопрос.
- Находит подходящие документы в базе.
- Отвечает, опираясь на эти документы.

**Главные плюсы**
1. **Актуальность** — базу знаний можно обновлять, и модель сразу использует свежие данные. Переобучать её не нужно.
2. **Прозрачность** — можно увидеть, откуда взят ответ, и проверить источники.
3. **Сильная сторона** — отлично работает с редкими и малоизвестными темами, где обычные модели часто ошибаются.

### Full Fine‑Tuning

**Что это:** модель полностью переобучается на новых данных. Меняются все параметры.

**Плюсы:**
- Максимальная точность (до 90% на специфических задачах).
- Глубокая настройка стиля и формата ответов.

**Минусы:**
- Очень дорого: для модели на 65 млрд параметров нужно ~780 ГБ видеопамяти.
- Знания застывают — при новых данных нужно переобучать заново.

**PEFT (эффективное дообучение):** меняется лишь 0,1–5% параметров модели, остальное остаётся без изменений.

**Методы:**
- **LoRA** — добавляет небольшие надстройки к весам.
- **QLoRA** — LoRA со сжатием модели.
- **Prompt Tuning** — обучаются «подсказки» вместо модели.
- **Prefix Tuning** — обучаются префиксы внутри модели.
- **Adapters** — маленькие обучаемые блоки между слоями.

**Плюсы:**
- Качество почти как у полного дообучения.
- Требует значительно меньше памяти и ресурсов.

**Минусы:**
- Чуть уступает полному дообучению в точности.

**Коротко:** полное дообучение — максимальное качество, но дорого. PEFT — почти то же качество, но значительно дешевле.

### Low‑Rank Adaptation (LoRA)

Это метод дообучения, при котором сама модель не меняется (её веса «замораживаются»). Вместо этого к ней добавляются небольшие корректирующие матрицы, которые и обучаются.

**Как это работает**

Обновление весов представляется как произведение двух маленьких матриц. Они значительно меньше исходной матрицы весов, поэтому обучать нужно в разы меньше параметров.

**Главные цифры**

- Количество обучаемых параметров сокращается до 10 000 раз (на примере GPT-3.5).
- Требования к памяти GPU снижаются в 3 раза.
- Качество ответов — почти как у полного дообучения.

**Где работает лучше всего**

Для моделей среднего размера — 7–13 млрд параметров — под конкретные задачи.

### Quantized Low‑Rank Adaptation (QLoRA)

Это усовершенствованная версия LoRA. Главная идея — сжать модель перед дообучением, чтобы она поместилась на обычную видеокарту.

**Три нововведения**

1. **NF4** — специальный 4-битный формат хранения весов, который лучше подходит для данных с нормальным распределением.
2. **Двойное квантование** — дополнительное сжатие для экономии памяти.
3. **Оптимизаторы с подкачкой страниц** — умное управление памятью при резких скачках нагрузки.

**Что это даёт**

- Модель с 65 млрд параметров теперь можно дообучать на обычном потребительском компьютере — раньше это было невозможно.
- Памяти нужно в 16 раз меньше, чем у обычного LoRA.
- Качество при этом почти не страдает.

### Prompt Tuning

Это метод, при котором сама модель остаётся полностью неизменной. Обучаются только специальные «мягкие промпты» — небольшие добавки к входным данным.

**Как это работает**

- Модель заморожена.
- Обучаются только эмбеддинги (векторы), которые добавляются к запросу.
- Эти обученные промпты помогают модели лучше понимать конкретную задачу.

**Плюсы**

- **Очень дёшево** — обучаются тысячи или миллионы параметров вместо миллиардов.
- **Одна модель — много задач** — можно создать разные «промпт-модули» для разных задач, не копируя модель.
- **Нет забывания** — модель не меняется, поэтому она не теряет прежние знания.

**Где полезно**

Когда одна модель должна выполнять много разных задач с минимальными затратами.

### Prefix Tuning

Это улучшенная версия Prompt Tuning. Обучаемые параметры добавляются не только на вход модели, но и в скрытые состояния каждого слоя трансформера.

**Отличие от Prompt Tuning**

- Prompt Tuning — меняет только входной слой.
- Prefix Tuning — влияет на все уровни модели, глубже настраивая её внутреннюю работу.

**Плюсы и минусы**

- **Плюс:** работает лучше, чем обычный Prompt Tuning, за счёт более глубокой адаптации.
- **Минус:** требует обучать больше параметров, чем Prompt Tuning, но всё равно значительно меньше, чем LoRA или полное дообучение.

**Коротко:** Prefix Tuning — это средний вариант между простым Prompt Tuning и LoRA: глубже, чем первый, но дешевле, чем второй.

### Адаптеры

Это маленькие дополнительные слои, которые вставляются внутрь модели — между блоками внимания и блоками прямой связи.

**Как устроены**

Каждый адаптер состоит из двух проекций (сжатие и расширение) с нелинейностью между ними. Это позволяет модели изучать новые параметры под конкретную задачу.

**Главный плюс — модульность**

- Для каждой задачи обучается свой набор адаптеров.
- Их можно легко менять местами или комбинировать.
- Не нужно переобучать всю модель.

**Сколько нужно ресурсов**

Обучается всего 1–5% от общего числа параметров модели.

**Коротко:** адаптеры — это съёмные «насадки» на модель. Каждая задача получает свою насадку, и их можно менять как конструктор.

### Сравнительная таблица подходов

**Что такое катастрофическое забывание**

Это проблема нейросетей: когда модель обучается на новых данных, она забывает то, что знала раньше. Причина — метод обучения (градиентный спуск) обновляет веса под текущие данные, не учитывая их важность для старых задач.

**Где это особенно критично**

В непрерывном обучении — когда данные поступают постепенно и постоянно меняются.

**Как с этим борются**

- **EWC** — защищает самые важные веса от изменений.
- **Повторное проигрывание** — сохраняются примеры из старых задач, и модель периодически на них «вспоминает».
- **Метапластичность** — подход, вдохновлённый работой мозга.
- **PEFT-методы** — модель остаётся замороженной, поэтому забывать нечему.

**Почему PEFT помогает**

Базовая модель не меняется. Для каждой задачи создаётся отдельный небольшой «адаптер», который можно подключать и отключать — без потери прежних знаний.

**Общий вывод**

Нет одного идеального метода. Выбор зависит от задачи, ресурсов и требований. Лучшие результаты часто дают гибридные подходы:

- **RAG** — для актуальности данных.
- **PEFT** — для специализации модели.

Их комбинация — современный стандарт.

| Характеристика | RAG | Full Fine-Tuning | LoRA | QLoRA | Prompt Tuning | Prefix Tuning | Адаптеры |
| --- | --- | --- | --- | --- | --- | --- | --- |
| Обучаемые параметры | 0 (модель не меняется) | 100 % параметров | 0,1–1 % параметров | 0,1–1 % параметров | < 0,01 % параметров | 0,01–0,1 % параметров | 1–5 % параметров |
| Требования к памяти GPU | Низкие (только inference) | Очень высокие (10–100+ Гб) | Средние (2–10 Гб) | Низкие (< 2 Гб) | Очень низкие (< 1 Гб) | Низкие (< 2 Гб) | Низкие (1–3 Гб) |
| Время обучения | Не требуется | Дни‑недели | Часы‑дни | Часы | Минуты‑часы | Часы | Часы‑дни |
| Актуальность данных | Высокая (обновление базы) | Низкая (требуется перетренировка) | Низкая | Низкая | Низкая | Низкая | Низкая |
| Точность | Средняя (40–68 %) | Очень высокая (80–91 %) | Высокая (75–88 %) | Высокая (70–85 %) | Средняя (50–70 %) | Средневысокая (60–75 %) | Высокая (70–85 %) |
| Прозрачность | Высокая (ссылки на источники) | Низкая | Низкая | Низкая | Низкая | Низкая | Низкая |
| Скорость инференса | Средняя (задержка на поиск) | Высокая | Высокая | Высокая | Высокая | Средняя | Средняя |
| Стоимость разработки | Низкая | Очень высокая | Средняя | Низкая | Очень низкая | Низкая | Средняя |
| Стоимость поддержки | Низкая | Высокая | Средняя | Средняя | Низкая | Низкая | Средняя |
| Специализация стиля | Низкая | Очень высокая | Высокая | Высокая | Средняя | Средняя | Высокая |
| Катастрофическое забывание | Отсутствует | Высокий риск | Средний риск | Средний риск | Низкий риск | Низкий риск | Средний риск |
| Многозадачность | Высокая | Низкая | Средняя | Средняя | Очень высокая | Высокая | Высокая |
| Размер артефактов | База знаний (Гб–Тб) | Полная модель (10–100+ Гб) | Адаптеры (10–100 Мб) | Адаптеры (5–50 Мб) | Промпты (< 1 Мб) | Префиксы (1–10 Мб) | Адаптеры (50–500 Мб) |